# 💡 Task 2: Restaurant Recommendation System
**Cognifyz Technologies — Machine Learning Internship**

---
**Objective:** Create a restaurant recommendation system based on user preferences.

**Steps:**
1. Preprocess the dataset by handling missing values and encoding categorical variables.
2. Determine the criteria for restaurant recommendations (e.g., cuisine preference, price range).
3. Implement a content-based filtering approach where users are recommended restaurants similar to their preferred criteria.
4. Test the recommendation system by providing sample user preferences and evaluating the quality of recommendations.

## 1. Import Libraries

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder

os.makedirs('plots', exist_ok=True)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('✅ Libraries loaded successfully!')

✅ Libraries loaded successfully!


## 2. Load Dataset

In [7]:
df = pd.read_csv('Dataset .csv')
df = df.reset_index(drop=True)
print(f'Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df[['Restaurant Name', 'City', 'Cuisines', 'Price range', 'Aggregate rating', 'Votes']].head()

Dataset Shape: 9,551 rows × 21 columns


,Restaurant Name,City,Cuisines,Price range,Aggregate rating,Votes
0,Le Petit Souffle,Makati City,"French, Japanese, Desserts",3,4.8,314
1,Izakaya Kikufuji,Makati City,Japanese,3,4.5,591
2,Heat - Edsa Shangri-La,Mandaluyong City,"Seafood, Asian, Filipino, Indian",4,4.4,270
3,Ooma,Mandaluyong City,"Japanese, Sushi",4,4.9,365
4,Sambo Kojin,Mandaluyong City,"Japanese, Korean",4,4.8,229


## 3. Step 1 — Preprocess: Handle Missing Values & Encode Categoricals

In [8]:
# Handle missing values
print('Missing values before:')
print(df.isnull().sum()[df.isnull().sum() > 0])

df['Cuisines'] = df['Cuisines'].fillna('Unknown')
df['City']     = df['City'].fillna('Unknown')
df['Locality'] = df['Locality'].fillna('Unknown')

print(f'\nMissing values after: {df.isnull().sum().sum()}')
print('✅ Missing values handled.')

Missing values before:
Cuisines    9
dtype: int64

Missing values after: 0
✅ Missing values handled.


In [9]:
# Encode binary categorical columns
binary_cols = ['Has Table booking', 'Has Online delivery', 'Is delivering now']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)

print('✅ Categorical variables encoded.')
print(df[binary_cols].value_counts().head(5))

✅ Categorical variables encoded.
Has Table booking  Has Online delivery  Is delivering now
0                  0                    0                    6377
                   1                    0                    1983
1                  0                    0                     723
                   1                    0                     434
0                  1                    1                      33
Name: count, dtype: int64


## 4. Step 2 — Determine Recommendation Criteria

In [ ]:
# Criteria used for recommendations:
# 1. Cuisine preference  (primary driver — TF-IDF)
# 2. Price range         (budget filter)
# 3. City / Locality     (location filter)
# 4. Table booking / delivery services (optional filters)

PRICE_LABELS = {1: '$ Budget', 2: '$$ Moderate', 3: '$$$ Expensive', 4: '$$$$ Luxury'}

print('=== Recommendation Criteria ===')
print('  1. Cuisine Preference  — encoded via TF-IDF on Cuisines + Locality')
print('  2. Price Range         — filter: 1 (Budget) to 4 (Luxury)')
print('  3. City                — optional location filter')
print('  4. Services            — table booking / online delivery flags')
print()

# Visualize price range distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pr_counts = df['Price range'].value_counts().sort_index()
colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
axes[0].bar([PRICE_LABELS[i] for i in pr_counts.index], pr_counts.values, color=colors, edgecolor='white')
for i, (_, val) in enumerate(pr_counts.items()):
    axes[0].text(i, val + 30, f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('Price Range Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)

# Top 10 cuisines
cuisine_counts = df['Cuisines'].str.split(', ').explode().value_counts().head(10)
axes[1].barh(cuisine_counts.index[::-1], cuisine_counts.values[::-1],
             color=sns.color_palette('husl', 10)[::-1])
axes[1].set_title('Top 10 Cuisines in Dataset', fontweight='bold')
axes[1].set_xlabel('Count')

plt.suptitle('Task 2: Dataset Overview for Recommendation Criteria', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0,0,1,0.95])
plt.savefig('plots/task2_criteria_overview.png', dpi=150, bbox_inches='tight')
plt.show()

=== Recommendation Criteria ===
  1. Cuisine Preference  — encoded via TF-IDF on Cuisines + Locality
  2. Price Range         — filter: 1 (Budget) to 4 (Luxury)
  3. City                — optional location filter
  4. Services            — table booking / online delivery flags



ValueError: 
$$ Moderate
^
ParseException: Expected end of text, found '$'  (at char 0), (line:1, col:1)

Error in callback <function _draw_all_if_interactive at 0x0000017BC9090700> (for post_execute), with arguments args (),kwargs {}:


ValueError: 
$$ Moderate
^
ParseException: Expected end of text, found '$'  (at char 0), (line:1, col:1)

ValueError: 
$$ Moderate
^
ParseException: Expected end of text, found '$'  (at char 0), (line:1, col:1)

<Figure size 1600x600 with 2 Axes>

## 5. Step 3 — Implement Content-Based Filtering

In [ ]:
# Build rich content string for each restaurant
def build_content(row):
    price_map = {
        1: 'budget cheap affordable',
        2: 'moderate mid-range',
        3: 'expensive premium fine-dining',
        4: 'luxury expensive elite'
    }
    price_str  = price_map.get(int(row['Price range']), 'moderate')
    cuisine_str = row['Cuisines'].replace(',', '').replace(' ', '_')
    city_str    = row['City'].replace(' ', '_')
    locality_str = row['Locality'].replace(' ', '_')

    content = f"{cuisine_str} {price_str} {city_str} {locality_str}"

    if row['Has Table booking'] == 1:
        content += ' table_booking reservations'
    if row['Has Online delivery'] == 1:
        content += ' online_delivery delivery'

    return content.lower()

df['content'] = df.apply(build_content, axis=1)

print('✅ Content strings built. Example:')
print(' ', df['content'].iloc[0])

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['content'])

print(f'✅ TF-IDF Matrix: {tfidf_matrix.shape[0]:,} restaurants × {tfidf_matrix.shape[1]:,} features')

# Compute cosine similarity matrix
print('Computing cosine similarity...')
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f'✅ Cosine Similarity Matrix: {cosine_sim.shape}')

In [ ]:
# --- Recommendation Function ---
def recommend_by_preference(cuisine, price_range, city=None, top_n=10, min_rating=0.0):
    """
    Recommend restaurants based on user preferences.
    
    Parameters:
    -----------
    cuisine    : str  — preferred cuisine (e.g., 'Italian', 'North Indian')
    price_range: int  — budget (1=Budget, 2=Moderate, 3=Expensive, 4=Luxury)
    city       : str  — optional city filter
    top_n      : int  — number of recommendations
    min_rating : float — minimum aggregate rating filter
    """
    price_map = {
        1: 'budget cheap affordable',
        2: 'moderate mid-range',
        3: 'expensive premium fine-dining',
        4: 'luxury expensive elite'
    }
    query = f"{cuisine.replace(' ', '_').lower()} {price_map.get(price_range, 'moderate')}"
    if city:
        query += f" {city.replace(' ', '_').lower()}"

    query_vec  = tfidf.transform([query])
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    sorted_idx = np.argsort(sim_scores)[::-1]

    # Apply filters
    mask = df['Price range'] == price_range
    if min_rating > 0:
        mask &= df['Aggregate rating'] >= min_rating
    if city:
        mask &= df['City'].str.lower().str.contains(city.lower(), na=False)

    filtered_idx = [i for i in sorted_idx if mask.iloc[i]]

    top_idx  = filtered_idx[:top_n]
    top_sims = sim_scores[top_idx]

    recs = df.loc[top_idx, ['Restaurant Name', 'City', 'Cuisines', 'Price range',
                             'Aggregate rating', 'Votes', 'Has Table booking', 'Has Online delivery']].copy()
    recs['Similarity Score'] = np.round(top_sims, 3)
    recs['Price Range'] = recs['Price range'].map(PRICE_LABELS)
    recs = recs.drop('Price range', axis=1).reset_index(drop=True)
    recs.index += 1
    return recs


def recommend_by_name(restaurant_name, top_n=10):
    """Recommend restaurants similar to a given restaurant name."""
    matches = df[df['Restaurant Name'].str.lower().str.contains(restaurant_name.lower(), na=False)]
    if matches.empty:
        print(f'❌ Restaurant "{restaurant_name}" not found!')
        return None
    idx  = matches.index[0]
    sims = list(enumerate(cosine_sim[idx]))
    sims = sorted(sims, key=lambda x: x[1], reverse=True)
    sims = [(i, s) for i, s in sims if i != idx][:top_n]
    top_idx  = [i for i, _ in sims]
    top_sims = [s for _, s in sims]
    recs = df.loc[top_idx, ['Restaurant Name', 'City', 'Cuisines', 'Price range',
                             'Aggregate rating', 'Votes']].copy()
    recs['Similarity'] = np.round(top_sims, 3)
    recs['Price Range'] = recs['Price range'].map(PRICE_LABELS)
    return recs.drop('Price range', axis=1).reset_index(drop=True)


print('✅ Content-based filtering functions ready!')

## 6. Step 4 — Test with Sample User Preferences

In [ ]:
# Test Case 1: Italian, Moderate budget
print('═' * 55)
print('  👤 User A: Italian cuisine, Moderate budget ($$$)')
print('═' * 55)
recs_a = recommend_by_preference('Italian', price_range=2, top_n=10)
display(recs_a.style.background_gradient(subset=['Aggregate rating'], cmap='YlGn').format({'Similarity Score': '{:.3f}'}))

In [ ]:
# Test Case 2: North Indian, Budget
print('═' * 55)
print('  👤 User B: North Indian, Budget ($)')
print('═' * 55)
recs_b = recommend_by_preference('North Indian', price_range=1, top_n=10)
display(recs_b.style.background_gradient(subset=['Aggregate rating'], cmap='YlGn').format({'Similarity Score': '{:.3f}'}))

In [ ]:
# Test Case 3: Chinese, Expensive, New Delhi
print('═' * 55)
print('  👤 User C: Chinese, Expensive ($$$), New Delhi')
print('═' * 55)
recs_c = recommend_by_preference('Chinese', price_range=3, city='New Delhi', top_n=10)
display(recs_c.style.background_gradient(subset=['Aggregate rating'], cmap='YlGn').format({'Similarity Score': '{:.3f}'}))

In [ ]:
# Test Case 4: By restaurant name
print('═' * 55)
print('  👤 User D: Show restaurants similar to "McDonald\'s"')
print('═' * 55)
recs_d = recommend_by_name("McDonald", top_n=10)
if recs_d is not None:
    display(recs_d.style.background_gradient(subset=['Aggregate rating'], cmap='YlGn'))

In [ ]:
# Quality evaluation — visualize recommendation ratings
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

test_cases = [
    ('User A: Italian/Moderate', recs_a, 'Aggregate rating'),
    ('User B: North Indian/Budget', recs_b, 'Aggregate rating'),
    ('User C: Chinese/Expensive/Delhi', recs_c, 'Aggregate rating'),
]

for ax, (label, recs, col) in zip(axes, test_cases):
    if recs is not None and len(recs) > 0:
        recs_plot = recs.head(10)
        colors = plt.cm.RdYlGn(recs_plot[col] / 5.0)
        bars = ax.barh(recs_plot['Restaurant Name'].str[:20][::-1],
                       recs_plot[col][::-1], color=colors[::-1], edgecolor='white')
        ax.set_xlim(0, 5.5)
        ax.set_title(label, fontsize=11, fontweight='bold')
        ax.set_xlabel('Rating')
        avg = recs_plot[col].mean()
        ax.axvline(avg, color='navy', linestyle='--', alpha=0.7, label=f'Avg: {avg:.2f}')
        ax.legend(fontsize=9)
    else:
        ax.text(0.5, 0.5, 'No results', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(label)

plt.suptitle('Task 2: Recommendation Quality — Ratings of Recommended Restaurants', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/task2_recommendation_quality.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Similarity heatmap — sample of top-rated restaurants
sample = df[df['Aggregate rating'] >= 4.5].head(12)
sample_idx = sample.index.tolist()
sim_sub = cosine_sim[np.ix_(sample_idx, sample_idx)]
labels  = [df.loc[i, 'Restaurant Name'][:18] for i in sample_idx]

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(sim_sub, xticklabels=labels, yticklabels=labels,
            cmap='Blues', annot=True, fmt='.2f', ax=ax,
            linewidths=0.5, square=True, annot_kws={'size': 9})
ax.set_title('Task 2: Cosine Similarity Among Top-Rated Restaurants', fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('plots/task2_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary

In [ ]:
print('=' * 65)
print('     TASK 2 SUMMARY — RESTAURANT RECOMMENDATION SYSTEM')
print('=' * 65)
print(f'📌 Total Restaurants     : {len(df):,}')
print(f'📌 TF-IDF Dimensions     : {tfidf_matrix.shape[1]:,} features')
print(f'📌 Similarity Matrix     : {cosine_sim.shape[0]:,} × {cosine_sim.shape[0]:,}')
print()
print('📋 Method: Content-Based Filtering')
print('   ▸ Content built from: Cuisines + Price level + City + Locality + Services')
print('   ▸ Vectorised using TF-IDF (1-gram and 2-gram)')
print('   ▸ Similarity measured via Cosine Similarity')
print()
print('🧪 Sample Tests:')
for label, recs in [('Italian/Moderate', recs_a), ('North Indian/Budget', recs_b),
                     ('Chinese/Expensive/Delhi', recs_c)]:
    avg_r = recs['Aggregate rating'].mean() if recs is not None and len(recs) > 0 else 0
    n     = len(recs) if recs is not None else 0
    print(f'   {label:25s}: {n} results | Avg rating = {avg_r:.2f}⭐')
print()
print('💡 Recommendation Criteria Used:')
print('   1. Cuisine preference  (TF-IDF weighted)')
print('   2. Price range filter  (exact match)')
print('   3. City filter         (optional)')
print('=' * 65)
print('✅ Task 2 Complete!')